# ResNet-18 on CIFAR-10 — activations in memory

`map()` with no path keeps activations in RAM. Downloads on first run:
CIFAR-10 test split (~20 MB) and ResNet-18 weights (~45 MB).

In [1]:
from torchvision.models import ResNet18_Weights, resnet18

from nnact import ActivationMapper
from nnact.utils import Cifar10Samples, activation_loader

dataset = Cifar10Samples(n=512)
print(f"{len(dataset)} samples | {tuple(dataset[0]['x'].shape)}")


512 samples | (3, 224, 224)


In [2]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)
mapper = ActivationMapper(model)

# Which layers can be hooked. depth=2 descends into the blocks; an unknown
# name raises from map() before the first forward pass, suggesting near misses.
mapper.summary(depth=1)

,module,parameters
layer,,
conv1,Conv2d,9408
bn1,BatchNorm2d,128
relu,ReLU,0
maxpool,MaxPool2d,0
layer1,Sequential,147968
layer2,Sequential,525568
layer3,Sequential,2099712
layer4,Sequential,8393728
avgpool,AdaptiveAvgPool2d,0


In [3]:
# The shared helper owns batching; the mapper receives its completed batches.
loader = activation_loader(dataset, batch_size=64)
result = mapper.map(loader, ["layer3", "layer4", "avgpool", "fc"])
result.activations.keys()


ResNet activations:   0%|          | 0/8 [00:00<?, ?it/s]

dict_keys(['layer3', 'layer4', 'avgpool', 'fc'])

In [4]:
# Per-sample shape per layer. layer3 is ~100x avgpool.
for name, tensor in result.activations.items():
    print(f"  {name:<10} {tuple(tensor.shape[1:])}")


  layer3     (256, 14, 14)
  layer4     (512, 7, 7)
  avgpool    (512, 1, 1)
  fc         (1000,)


In [5]:
# One stacked tensor per layer, (n_samples, *act_shape), in loader order.
print("avgpool stacked:", tuple(result.activations["avgpool"].shape))


avgpool stacked: (512, 512, 1, 1)


In [6]:
# Layer choice dominates storage. One sample is enough to read the shapes
# and project the cost at any dataset size.
probe_loader = activation_loader(Cifar10Samples(n=1), batch_size=1)
probe = mapper.map(
    probe_loader,
    mapper.available_layers(depth=1),
    progress=False,
)

for name, tensor in probe.activations.items():
    elements = tensor.shape[1:].numel()
    print(
        f"  {name:<10} {tuple(tensor.shape[1:])!s:<20} "
        f"{elements * 4 / 1024:8.2f} KB/sample  "
        f"{elements * 4 * 10_000 / 1024**3:6.2f} GB/10k imgs"
    )


  conv1      (64, 112, 112)        3136.00 KB/sample   29.91 GB/10k imgs
  bn1        (64, 112, 112)        3136.00 KB/sample   29.91 GB/10k imgs
  relu       (64, 112, 112)        3136.00 KB/sample   29.91 GB/10k imgs
  maxpool    (64, 56, 56)           784.00 KB/sample    7.48 GB/10k imgs
  layer1     (64, 56, 56)           784.00 KB/sample    7.48 GB/10k imgs
  layer2     (128, 28, 28)          392.00 KB/sample    3.74 GB/10k imgs
  layer3     (256, 14, 14)          196.00 KB/sample    1.87 GB/10k imgs
  layer4     (512, 7, 7)             98.00 KB/sample    0.93 GB/10k imgs
  avgpool    (512, 1, 1)              2.00 KB/sample    0.02 GB/10k imgs
  fc         (1000,)                  3.91 KB/sample    0.04 GB/10k imgs
